# Interactive Time-Series Explorer

Pick a **dataset** (soil or met), a **station**, then a **variable** to see its
hourly time series. The dropdowns cascade: changing the dataset refills the
station list, and changing the station refills the variable list.

Wherever the **selected variable has no measurement (`NaN`)**, that stretch of
time is shaded as a **red block** on the plot. Because the data is pre-washed
(`prewash_the_data=True`), this captures both filled-in missing timestamps
*and* replaces impossible values with NA

### Requirements
Needs `plotly` and `ipywidgets`.

In [1]:
import os, sys
from pathlib import Path

# Directory containing this notebook.
NOTEBOOK_DIR = Path.cwd()

# Add the scripts folder to the import path, then import data_ingest.
sys.path.insert(0, str(NOTEBOOK_DIR / "scripts"))
from get_data_dict import data_ingest

# Raw TxSON data folder: the TXSON_DATA env var, or a default sibling path.
DATA_FOLDER = str(Path(os.environ.get(
    "TXSON_DATA",
    NOTEBOOK_DIR.parent / "datasets" / "TxSON_data_2026-02-24",
)))

In [2]:
# prewash_the_data fills missing hourly timestamps with NaN so gaps show as red blocks.
ingest = data_ingest(
    input_data_folder=DATA_FOLDER,
    prewash_the_data=True,
    clean_the_data=False,
    # download = True,
    # prewash_folder=None, # where to download prewash data. If None, it will be set to "prewashed_data" in the working directory
)
met_dict, soil_dict = ingest.get_data_dict()

print(f"Met  stations: {len(met_dict):>2}  ->  {list(met_dict)}")
print(f"Soil stations: {len(soil_dict):>2}  ->  {list(soil_dict)}")

Met  stations:  6  ->  ['CB01_met', 'CB04_met', 'CB06_met', 'FD02_met', 'FD03_met', 'WC05_met']
Soil stations: 33  ->  ['CB01', 'CB04', 'CB06', 'CB07', 'CB09', 'CB10', 'CB15', 'CB19', 'CB20', 'CB25', 'CB26', 'CB27', 'CB31', 'CB32', 'CB33', 'FD02', 'FD03', 'FD08', 'FD11', 'FD12', 'FD13', 'FD14', 'FD16', 'FD17', 'FD18', 'FD21', 'FD22', 'FD23', 'FD24', 'FD28', 'FD29', 'FD30', 'WC05']


# AI CODE

In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Two source dictionaries keyed by station name. "Flag" is not a measurement.
DATASETS = {"Soil": soil_dict, "Met": met_dict}
IGNORE_COLS = {"Flag"}
LINE_COLOR = "#1f77b4"

# NA gaps are shaded by duration with four distinct colors (green -> dark red).
GAP_ORDER = ["short", "medium", "long", "very_long"]
NA_OPACITY = 0.8  # one shading opacity for every bucket, so light colors read
GAP_COLORS = {"short": "#2ca02c", "medium": "#ff7f0e",
              "long": "#d62728", "very_long": "#7f0000"}
GAP_LABELS = {"short": "<24h", "medium": "1–7d",
              "long": "7–30d", "very_long": ">30d"}


def na_blocks(mask, index):
    """Group a boolean NA mask into contiguous (start, end) timestamp spans."""
    m = np.asarray(mask, dtype=bool)
    if not m.any():
        return []
    edges = np.diff(np.concatenate(([0], m.astype(np.int8), [0])))
    starts = np.where(edges == 1)[0]
    ends = np.where(edges == -1)[0] - 1  # inclusive end of each run
    return [(index[s], index[e]) for s, e in zip(starts, ends)]


def gap_category(span):
    """Bucket one gap duration into a severity key: short/medium/long/very_long."""
    day = pd.Timedelta(days=1)
    if span < day:
        return "short"
    if span < 7 * day:
        return "medium"
    if span <= 30 * day:
        return "long"
    return "very_long"


def classify_gaps(blocks, step):
    """Count NA gaps per severity bucket (gap span = missing samples x step)."""
    counts = {k: 0 for k in GAP_ORDER}
    for start, end in blocks:
        counts[gap_category((end - start) + step)] += 1
    return counts


# ---- figure ---------------------------------------------------------------
# trace 0 is the data line (updated in _draw); traces 1-4 are fixed legend
# swatches, one per gap-length bucket.
fig = go.FigureWidget()
fig.add_scatter(mode="lines", line=dict(color=LINE_COLOR, width=1),
                connectgaps=False, showlegend=False)  # trace 0: the data line
for _key in GAP_ORDER:
    fig.add_scatter(x=[None], y=[None], mode="markers",
                    marker=dict(symbol="square", size=11,
                                color=GAP_COLORS[_key], opacity=NA_OPACITY),
                    name=f"NA {GAP_LABELS[_key]}")
fig.update_layout(
    template="plotly_white", height=480,
    margin=dict(l=60, r=20, t=92, b=40), hovermode="x unified",
    xaxis_title="Date", title=dict(x=0.5, xanchor="center"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
last_fig = fig  # the live figure, handy for exporting the current view


def _draw(dataset, station, variable, df):
    """Update the figure for one variable; NA stretches shaded by gap length."""
    series = pd.to_numeric(df[variable], errors="coerce")
    mask = series.isna()
    blocks = na_blocks(mask.to_numpy(), df.index)

    # Representative sampling step (hourly data); used for gap length + padding.
    if len(df.index) > 1:
        step = pd.Timedelta(np.median(np.diff(df.index.values)))
    else:
        step = pd.Timedelta(hours=1)
    pad = step / 2  # widen each block so single-sample gaps stay visible

    shapes = [
        dict(type="rect", xref="x", yref="paper",
             x0=start - pad, x1=end + pad, y0=0, y1=1,
             fillcolor=GAP_COLORS[gap_category((end - start) + step)],
             opacity=NA_OPACITY, line_width=0, layer="below")
        for start, end in blocks
    ]

    g = classify_gaps(blocks, step)
    n_na, n = int(mask.sum()), len(series)
    pct = (100 * n_na / n) if n else 0.0
    title = (
        f"{dataset} · {station} · {variable}   —   "
        f"{n_na:,} NA / {n:,} ({pct:.1f}%) in {len(blocks):,} gap(s)"
    )

    # Update the data line in place and apply the layout in a single batch, so the
    # figure redraws once (fast, and no red flash from a half-updated figure).
    with fig.batch_update():
        fig.data[0].x = df.index
        fig.data[0].y = series.to_numpy()
        fig.data[0].name = variable
        for i, key in enumerate(GAP_ORDER, start=1):   # swatch labels carry counts
            fig.data[i].name = f"NA {GAP_LABELS[key]}: {g[key]}"
        fig.layout.shapes = shapes
        fig.layout.title.text = title
        fig.layout.yaxis.title.text = variable


# ---- controls -------------------------------------------------------------
dataset_dd  = widgets.Dropdown(options=list(DATASETS), description="Dataset:")
station_dd  = widgets.Dropdown(description="Station:")
variable_dd = widgets.Dropdown(description="Variable:")
_updating   = {"on": False}  # suppress handlers during programmatic option changes


def _redraw(*_):
    ds, station, variable = dataset_dd.value, station_dd.value, variable_dd.value
    df = DATASETS.get(ds, {}).get(station)
    if df is None or variable is None or variable not in df.columns:
        return
    _draw(ds, station, variable, df)


def _sync_variable_options():
    df = DATASETS.get(dataset_dd.value, {}).get(station_dd.value)
    cols = [c for c in df.columns if c not in IGNORE_COLS] if df is not None else []
    _updating["on"] = True
    variable_dd.options = cols
    variable_dd.value = cols[0] if cols else None
    _updating["on"] = False


def _sync_station_options():
    keys = list(DATASETS.get(dataset_dd.value, {}))
    _updating["on"] = True
    station_dd.options = keys
    station_dd.value = keys[0] if keys else None
    _updating["on"] = False


def _on_dataset_change(*_):
    if _updating["on"]:
        return
    _sync_station_options()
    _sync_variable_options()
    _redraw()


def _on_station_change(*_):
    if _updating["on"]:
        return
    _sync_variable_options()
    _redraw()


def _on_variable_change(*_):
    if _updating["on"]:
        return
    _redraw()


dataset_dd.observe(_on_dataset_change, names="value")
station_dd.observe(_on_station_change, names="value")
variable_dd.observe(_on_variable_change, names="value")

# Draw the first view before display so the traces are part of the figure's
# initial state.
_sync_station_options()
_sync_variable_options()
_redraw()

# Show the controls, then the figure as its own top-level output. A FigureWidget
# nested in a VBox/HBox renders blank until the first resize; shown standalone it
# draws its initial traces immediately.
display(widgets.HBox([dataset_dd, station_dd, variable_dd]))
display(fig)

FigureWidget({
    'data': [{'connectgaps': False,
              'line': {'color': '#1f77b4', 'width': 1},
              'mode': 'lines',
              'name': 'Ppt',
              'showlegend': False,
              'type': 'scatter',
              'uid': '015cf669-c100-4f1e-b4a7-6c49a5ca5406',
              'x': array(['2014-09-29T18:00:00.000000', '2014-09-29T19:00:00.000000',
                          '2014-09-29T20:00:00.000000', ..., '2024-03-19T06:00:00.000000',
                          '2024-03-19T07:00:00.000000', '2024-03-19T08:00:00.000000'],
                         shape=(83007,), dtype='datetime64[us]'),
              'y': {'bdata': ('exSuR+F6H0AAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA'),
                    'dtype': 'f8'}},
             {'marker': {'color': '#2ca02c', 'opacity': 0.8, 'size': 11, 'symbol': 'square'},
              'mode': 'markers',
              'name': 'NA <24h: 2',
              'type': 'scatter',
              'uid': 'edfd73d2-e031-44d